# Business Insights Analytics Project

This project demonstrates a typical workflow for a business analytics or data analyst role. We work with a synthetic dataset that simulates program and project management scenarios. The notebook performs exploratory data analysis (EDA), visualizations, and builds predictive models.

## Dataset description

The dataset (`synthetic_projects.csv`) contains simulated project-level data to mimic real-world program management scenarios. Columns include:

- `project_id`: Unique identifier for each project.
- `industry`: Industry sector of the project (Finance, Healthcare, Technology, Retail, Manufacturing).
- `team_size`: Number of team members assigned to the project.
- `budget_k`: Budget allocated for the project (in thousands of dollars).
- `duration_months`: Expected duration of the project in months.
- `complexity`: A score (1–10) representing project complexity.
- `vendor_count`: Number of external vendors involved.
- `risk_score`: Risk score (0–1), where higher values indicate more risk.
- `stakeholder_engagement`: Stakeholder engagement level (1–5).
- `actual_cost_k`: Actual cost incurred (in thousands of dollars).
- `success`: Binary indicator (1 if the project met cost and risk criteria, 0 otherwise).
- `satisfaction`: Stakeholder satisfaction score (0–10).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Load dataset
file_path = '../data/synthetic_projects.csv'
df = pd.read_csv(file_path)

# Display first few rows
df.head()


In [ ]:
# Basic information and summary statistics
df.info()
df.describe(include='all')


## Exploratory data analysis and visualizations

In [ ]:
sns.set(style='whitegrid', palette='viridis')

# Distribution of budget
plt.figure(figsize=(6,4))
sns.histplot(df['budget_k'], bins=30, kde=True)
plt.title('Distribution of Project Budget (k)')
plt.xlabel('Budget (k)')
plt.ylabel('Frequency')
plt.show()

# Team size vs budget
plt.figure(figsize=(6,4))
sns.scatterplot(x='team_size', y='budget_k', hue='industry', data=df, alpha=0.7)
plt.title('Team Size vs Budget by Industry')
plt.xlabel('Team Size')
plt.ylabel('Budget (k)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Correlation heatmap
corr = df[['team_size','budget_k','duration_months','complexity','vendor_count','risk_score','actual_cost_k','satisfaction']].corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()


## Predictive modeling: Project success classification

We build models to predict whether a project will be successful based on its features.

In [ ]:
# Prepare features and target
X = df[['industry','team_size','budget_k','duration_months','complexity','vendor_count','risk_score','stakeholder_engagement']]
y = df['success']

# Identify categorical and numeric columns
categorical_cols = ['industry']
numeric_cols = ['team_size','budget_k','duration_months','complexity','vendor_count','risk_score','stakeholder_engagement']

# Preprocess: one-hot encode categorical variables and scale numeric variables (if needed)
preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(drop='first'), categorical_cols),
        ('numeric', 'passthrough', numeric_cols)
    ])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Model 1: Logistic Regression
log_reg = Pipeline(steps=[('preprocessor', preprocessor),
                     ('classifier', LogisticRegression(max_iter=500))])
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)

print("Logistic Regression Classification Report:
", classification_report(y_test, y_pred_lr))
print("Confusion Matrix:
", confusion_matrix(y_test, y_pred_lr))

# Model 2: Random Forest
rf = Pipeline(steps=[('preprocessor', preprocessor),
                    ('classifier', RandomForestClassifier(n_estimators=200, random_state=42))])
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Classification Report:
", classification_report(y_test, y_pred_rf))
print("Confusion Matrix:
", confusion_matrix(y_test, y_pred_rf))


### Interpretation

The logistic regression model provides an interpretable baseline for predicting project success. The random forest model may capture nonlinear relationships and interactions between features, often improving performance. Depending on your use case, you might further tune hyperparameters, evaluate additional models, or analyze feature importances.